# Password Storage and Cracking

## Goal

This notebook shows why authentication systems should not store plaintext passwords and why salted,
slow password verifiers are better than simple hashes.

The setting is St. Isidore Hospital. We create a few toy staff accounts, then simulate an attacker
who obtains a password file. This is a teaching model only. Real systems use mature identity
providers, password hashing libraries, MFA platforms, monitoring, and recovery procedures.

In [ ]:
import hashlib
import hmac
import os
import time


# St. Isidore toy users. Some passwords are intentionally weak for the attack demo.
users = {
    "dr.moretti": "StIsidore2026!",
    "nurse.bianchi": "blue-river-calm-ward",
    "pharmacist.galli": "Pharmacy123!",
    "patient.rossi": "maria1968",
}

print("Toy users:", list(users))

## Bad design: plaintext password storage

Plaintext storage is dangerous because a database leak immediately becomes an account compromise.
The attacker does not need to crack anything.

In [ ]:
# This is intentionally bad. Do not store passwords like this.
plaintext_password_file = users.copy()

for username, password in plaintext_password_file.items():
    print(f"{username:18s} -> {password}")

## Better design: salted PBKDF2 verifiers

The system stores a random salt and a derived verifier. At login, it recomputes the verifier from
the submitted password and compares it in constant time.

In [ ]:
def make_verifier(password: str, iterations: int = 200_000) -> dict:
    # Generate a different salt for every password.
    salt = os.urandom(16)

    # Derive a slow password verifier with PBKDF2-HMAC-SHA256.
    verifier = hashlib.pbkdf2_hmac(
        "sha256",
        password.encode("utf-8"),
        salt,
        iterations,
    )

    # Store parameters needed for verification, not the plaintext password.
    return {"salt": salt, "iterations": iterations, "verifier": verifier}


def verify_password(password: str, record: dict) -> bool:
    # Recompute the verifier using the stored salt and iteration count.
    candidate = hashlib.pbkdf2_hmac(
        "sha256",
        password.encode("utf-8"),
        record["salt"],
        record["iterations"],
    )

    # Compare securely to avoid timing leaks in real implementations.
    return hmac.compare_digest(candidate, record["verifier"])


password_file = {username: make_verifier(password) for username, password in users.items()}

for username, record in password_file.items():
    print(username, record["salt"].hex(), record["verifier"].hex()[:24] + "...")

In [ ]:
# A legitimate login recomputes the verifier and compares it with the stored value.
print("Correct password:", verify_password("blue-river-calm-ward", password_file["nurse.bianchi"]))
print("Wrong password:  ", verify_password("blue-river-calm-ward!", password_file["nurse.bianchi"]))

## Offline dictionary attack

If an attacker steals the password file, they can test guesses offline. Salts do not stop guessing,
but they prevent simple reuse of precomputed tables and force account-specific work.

In [ ]:
# A small attacker dictionary. Real dictionaries contain millions or billions of guesses.
dictionary = [
    "admin",
    "password",
    "Password123!",
    "StIsidore2026!",
    "Pharmacy123!",
    "maria1968",
    "blue-river-calm-ward",
    "correct-horse-battery-staple",
]


def crack_account(username: str, record: dict, guesses: list[str]) -> str | None:
    # Try each candidate against the stolen verifier.
    for guess in guesses:
        if verify_password(guess, record):
            return guess
    return None


for username, record in password_file.items():
    cracked = crack_account(username, record, dictionary)
    print(f"{username:18s} -> {cracked}")

## Cost matters

Slow hashing increases the cost of each guess. Administrators must choose parameters that fit their
systems and policies. Too cheap helps attackers; too expensive can harm login availability.

In [ ]:
def time_one_guess(iterations: int) -> float:
    # Build a record with the chosen iteration count.
    record = make_verifier("test-password", iterations=iterations)

    # Measure one verification attempt.
    start = time.perf_counter()
    verify_password("wrong-password", record)
    return time.perf_counter() - start


for iterations in [10_000, 50_000, 200_000]:
    elapsed = time_one_guess(iterations)
    print(f"{iterations:>7,d} iterations -> {elapsed:.4f} seconds per guess")

## Takeaway

Salted slow verifiers reduce the damage of a stolen password file, but they do not make weak
passwords safe. St. Isidore still needs password screening, rate limits, MFA for sensitive systems,
monitoring, and secure recovery.